In [2]:
import os
import csv
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ====== Config ======
IN_PATH  = "jira_TD_dataset_clean.csv"
MODEL_NAME = "karths/binary_classification_train_TD"
TEXT_COL = "testo"                 # OK: nel tuo CSV c'è "testo"
ID_COL = "numero_frase"            # <-- aggiunto
MAX_LENGTH_TOK = 256
N_FIRST = None
SAVE_EVERY = 10

OUT_FILE = "risultati_jira-trans-token.csv"
SKIPPED_FILE = "skipped_rows.csv"

# ====== Load teacher ======
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# ====== Load dataset ======
df = pd.read_csv(IN_PATH)
if N_FIRST is not None:
    df = df.head(N_FIRST)
rows = df.to_dict(orient="records")

# ====== File init (append-safe) ======
def init_csv(path, header):
    exists = os.path.exists(path)
    f = open(path, "a", newline="", encoding="utf-8")
    writer = csv.DictWriter(f, fieldnames=header)
    if not exists:
        writer.writeheader()
    return f, writer

out_f, out_writer = init_csv(
    OUT_FILE,
    ["numero_frase", "label", "orig_text", "p_td", "logit_td", "tokens", "token_scores"]
)

skip_f, skip_writer = init_csv(
    SKIPPED_FILE,
    ["numero_frase", "reason", "orig_text"]
)

# ====== Helper: token attributions (Grad × Input) ======
def token_attributions_grad_x_input(text: str):
    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH_TOK,
        return_tensors="pt"
    ).to(device)

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    embeddings = model.roberta.embeddings.word_embeddings

    model.zero_grad(set_to_none=True)

    inputs_embeds = embeddings(input_ids)
    inputs_embeds.retain_grad()          # fondamentale
    inputs_embeds.requires_grad_(True)

    outputs = model(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask
    )

    logits = outputs.logits
    # TD = classe 1 (per roberta sequence classification è quasi sempre così)
    if logits.shape[-1] == 1:
        logit_td = logits.squeeze(-1)
        p_td = torch.sigmoid(logits).squeeze(-1)
    else:
        logit_td = logits[:, 1]
        p_td = torch.softmax(logits, dim=-1)[:, 1]

    logit_td.backward()

    grads = inputs_embeds.grad[0]
    embeds = inputs_embeds.detach()[0]
    token_scores = (grads * embeds).sum(dim=1).abs().detach().cpu().tolist()

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].detach().cpu())

    return tokens, token_scores, float(p_td.item()), float(logit_td.item())

# ====== Main loop ======
buffer = []
processed = 0

for i, row in enumerate(rows):
    frase_id = row.get(ID_COL, i)         # <-- usa numero_frase
    text = row.get(TEXT_COL, None)
    label = row.get("label", None)

    if not isinstance(text, str) or not text.strip():
        skip_writer.writerow({
            "numero_frase": frase_id,
            "reason": "empty text",
            "orig_text": "" if text is None else str(text)
        })
        continue

    try:
        tokens, token_scores, p_td, logit_td = token_attributions_grad_x_input(text)
    except Exception as e:
        skip_writer.writerow({
            "numero_frase": frase_id,
            "reason": f"exception: {type(e).__name__}: {e}",
            "orig_text": text
        })
        continue

    buffer.append({
        "numero_frase": frase_id,
        "label": int(label) if label is not None and str(label).strip() != "" else None,
        "orig_text": text,
        "p_td": p_td,
        "logit_td": logit_td,
        "tokens": json.dumps(tokens, ensure_ascii=False),
        "token_scores": json.dumps(token_scores, ensure_ascii=False),
    })

    processed += 1

    if processed % SAVE_EVERY == 0:
        for r in buffer:
            out_writer.writerow(r)
        out_f.flush()
        buffer.clear()
        print(f"[checkpoint] salvate {processed} righe")

# flush finale
for r in buffer:
    out_writer.writerow(r)
out_f.flush()

out_f.close()
skip_f.close()

print("FINE. Righe processate:", processed)

c:\Users\o.sacilotto\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\o.sacilotto\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\o.sacilotto\.cache\huggingface\hub\models--karths--binary_classification_train_TD. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Develo

[checkpoint] salvate 10 righe
[checkpoint] salvate 20 righe
[checkpoint] salvate 30 righe
[checkpoint] salvate 40 righe
[checkpoint] salvate 50 righe
[checkpoint] salvate 60 righe
[checkpoint] salvate 70 righe
[checkpoint] salvate 80 righe
[checkpoint] salvate 90 righe
[checkpoint] salvate 100 righe
[checkpoint] salvate 110 righe
[checkpoint] salvate 120 righe
[checkpoint] salvate 130 righe
[checkpoint] salvate 140 righe
[checkpoint] salvate 150 righe
[checkpoint] salvate 160 righe
[checkpoint] salvate 170 righe
[checkpoint] salvate 180 righe
[checkpoint] salvate 190 righe
[checkpoint] salvate 200 righe
[checkpoint] salvate 210 righe
[checkpoint] salvate 220 righe
[checkpoint] salvate 230 righe
[checkpoint] salvate 240 righe
[checkpoint] salvate 250 righe
[checkpoint] salvate 260 righe
[checkpoint] salvate 270 righe
[checkpoint] salvate 280 righe
[checkpoint] salvate 290 righe
[checkpoint] salvate 300 righe
[checkpoint] salvate 310 righe
[checkpoint] salvate 320 righe
[checkpoint] salv